# Notebook 10: Sensor Failure & Missing-Modality Robustness Study

**Objective**: Quantify model resilience against noisy, degraded, or failing wearable sensor channels.

### Experimental Scenarios:
1. **Random Channel Dropout**: 10%, 25%, 50%, and 75% of on-body channels randomly masked.
2. **Complete Regional Outages**: Total loss of individual body regions (Trunk, Right Arm, Left Arm, Right Leg, Left Leg).

**Compute Environment**: CPU Safe.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT DETECTION & REPOSITORY BOOTSTRAP
# ==============================================================================
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("[INFO] Cloning Opportunity_HAR repository from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    else:
        print(f"[INFO] Repository already present at {REPO_DIR}. Pulling latest changes...")
        !git -C {REPO_DIR} pull origin main

    # Set repository root at front of sys.path
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

print(f"Active Working Directory : {Path.cwd()}")

# Ensure fresh import of src.api
import importlib
if "src.api" in sys.modules:
    import src.api
    importlib.reload(src.api)
else:
    import src.api as api

import src.api as api
status = api.setup_project()
print(f"High-Level API Loaded    : src.api (locate_dataset available: {hasattr(api, 'locate_dataset')})")
print(f"Compute Device Detected  : {status['device_name']} (CUDA Available: {status['cuda_available']})")

## 1. Execute Robustness Evaluation (CPU)

In [ ]:
try:
    robustness_df = api.run_robustness(model_type="random_forest")
    print("\n" + "=" * 75)
    print(" SENSOR FAILURE & MISSING-MODALITY ROBUSTNESS RESULTS")
    print("=" * 75)
    print(robustness_df[["Failure Scenario", "Dropped Channels", "Accuracy", "Macro F1", "Degradation (Macro F1)"]].to_string(index=False))
    print("=" * 75)
except Exception as e:
    print("Robustness execution notice:", e)
    print("Ensure preprocessed dataset is available (Notebook 02).")

## 2. Visualize Failure Degradation Curves

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Panel A: Random Dropout Curve
    rand_df = robustness_df[robustness_df["Condition Type"] == "Random Dropout"].copy()
    drop_pcts = [0, 10, 25, 50, 75]
    axes[0].plot(drop_pcts, rand_df["raw_macro_f1"] * 100, marker="o", linewidth=2.5, color="#1f77b4")
    axes[0].set_title("Performance vs. Random Sensor Channel Loss")
    axes[0].set_xlabel("Percentage of Randomly Dropped Channels (%)")
    axes[0].set_ylabel("Macro F1 Score (%)")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    # Panel B: Regional Outages
    reg_df = robustness_df[robustness_df["Condition Type"] == "Regional Failure"].copy()
    sns.barplot(data=reg_df, x="Failure Scenario", y="raw_macro_f1", ax=axes[1], palette="Reds_r")
    axes[1].set_title("Degradation Under Complete Regional Outages")
    axes[1].set_ylabel("Macro F1 Score")
    axes[1].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Plotting notice:", e)